In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("volume", "")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()

# Perfiles conocidos: se elige el primero cuyo catálogo exista en este workspace
PERFILES = [
    {"catalog": "oceanwatch_e2", "schema": "",         "volume": ""},             # Free Edition
    {"catalog": "clase10",       "schema": "equipo10", "volume": "checkpoints"},  # Workspace institucional
]
catalogos = {r[0] for r in spark.sql("SHOW CATALOGS").collect()}

# Si no se pasó catálogo, o el que se pasó no existe aquí, se autodetecta el perfil
if not catalog or catalog not in catalogos:
    perfil = next((p for p in PERFILES if p["catalog"] in catalogos), None)
    if perfil is None:
        raise Exception(f"No se reconoce el workspace: ninguno de {[p['catalog'] for p in PERFILES]} existe")
    catalog, schema, volume = perfil["catalog"], perfil["schema"], perfil["volume"]

def T(capa, nombre):
    """Nombre completo de tabla según el modo."""
    return f"{catalog}.{schema}.{capa}_{nombre}" if schema else f"{catalog}.{capa}.{nombre}"

_CARPETAS = {"raw_zips": ("landing", "raw_zips"), "csv": ("landing", "csv"),
             "ref": ("landing", "ref"), "checkpoints": ("ops", "checkpoints"),
             "entrada": ("landing", "entrada")}

def V(carpeta):
    """Ruta en Volume según el modo (raw_zips, csv, ref, checkpoints)."""
    if volume:
        p = f"/Volumes/{catalog}/{schema}/{volume}/oceanwatch/{carpeta}"
    else:
        esq, vol = _CARPETAS[carpeta]
        p = f"/Volumes/{catalog}/{esq}/{vol}"
    dbutils.fs.mkdirs(p)
    return p

print(f"catalog: {catalog} | schema: {schema or '(uno por capa)'} | volume: {volume or '(propios)'}")